# ContextAgent: from `ts_check` diagnostics to a trained AutoML model

The user never mentions `config_dict` or its schema — they just say what they want in natural language. Under the hood the ContextAgent chains:

1. **`ts_check`** to profile the series,
2. **`get_automl_config_dict`** to fetch PAL's `default`/`light`/`empty` template for the pipeline as JSON,
3. **`modify_automl_config_dict`** (with `verify=True`) if the agent wants to add / remove / replace / modify operators — PAL's own `VERIFY_CONFIG=1` gates the schema,
4. **`automatic_timeseries_fit_and_save`** with the resolved `config_dict`,
5. **`automatic_timeseries_load_model_and_predict`** + **`forecast_line_plot`** for the forecast.

The tools wrap `_SYS_AFL.PAL_PIPELINE_INFO` and `_SYS_AFL.PAL_AUTOML_CONFIG` directly, so the operator catalogue and every validation call reflect what the target HANA instance actually supports. See [SAP Help — Pipeline Operator](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/pipeline-operator-pipeline-operator).

## 1. Prepare data and the ContextAgent

In [ ]:
import certifi
import os
from hana_ml import dataframe

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
# Demo-only: force the AutoML fit tools to refuse a bare call and route the
# agent through the ts_check -> get_automl_config_dict -> optional
# modify_automl_config_dict -> fit flow. Default production behaviour (bare
# call runs the PAL default pipeline) is unchanged when this env var is not
# set.
os.environ["HANA_AI_AUTOML_REQUIRE_CONFIG_DICT"] = "1"

cc = dataframe.ConnectionContext(
    userkey="RaysKey",
    sslValidateCertificate=False,
    encrypt=True,
    sslKeyStore=certifi.where(),
)

In [ ]:
import importlib
import uuid

from gen_ai_hub.proxy.langchain import init_llm
import hana_ai.iagents.context_agent as context_agent_module
from hana_ai.tools.toolkit import HANAMLToolkit

context_agent_module = importlib.reload(context_agent_module)
ContextAgent = context_agent_module.ContextAgent
AgentConfig = context_agent_module.AgentConfig

llm = init_llm("gpt-4.1")

# HANAMLToolkit registers ts_check, get_pal_pipeline_info, get_automl_config_dict,
# modify_automl_config_dict, and the AutoML tools by default.
tools = HANAMLToolkit(cc, used_tools="all").get_tools()

# ---- Trace every tool call so we can see exactly what the agent did. ----
# We shim BaseTool._run for every registered tool. The agent still sees normal
# LangChain tools; nothing about routing changes.
tool_call_log = []


def _install_tracer(tool_obj):
    original_run = tool_obj._run

    def traced_run(**kwargs):
        payload = kwargs.get("kwargs", kwargs)
        entry = {
            "tool": tool_obj.name,
            "inputs": {k: v for k, v in payload.items() if k != "kwargs"},
        }
        try:
            result = original_run(**kwargs)
        except Exception as exc:
            entry["error"] = repr(exc)
            tool_call_log.append(entry)
            raise
        import json as _json
        try:
            entry["result"] = _json.loads(result) if isinstance(result, str) else result
        except Exception:
            entry["result"] = result
        tool_call_log.append(entry)
        return result

    tool_obj._run = traced_run


for t in tools:
    _install_tracer(t)

chatbot = ContextAgent(
    llm=llm,
    session_id="tscheck-automl-" + str(uuid.uuid1()),
    tools=tools,
    progress_bar=True,
    config=AgentConfig(skills_use_llm_selector=True, max_active_skills=4),
)
chatbot.chat("!reset_memory")

In [ ]:
print(chatbot.chat(
    "Upload SALES_REFUNDS.csv into a table named SALES_REFUNDS in HANA. "
    "Columns: BOOKING_DATE (YYYY-MM-DD), REFUNDS (numeric). Overwrite if it exists. "
    "Then split it in time order into SALES_REFUNDS_TRAIN and SALES_REFUNDS_PREDICT."
))

## 2. Profile the training series

One natural-language question — no talk of algorithms, no `config_dict`. The agent picks `ts_check`.

In [ ]:
print(chatbot.chat(
    "What are the key characteristics of SALES_REFUNDS_TRAIN? "
    "BOOKING_DATE is the time key and REFUNDS is the target."
))

In [ ]:
# Recast BOOKING_DATE from LONGDATE to DATE
cc.sql("SELECT DATA_TYPE_NAME FROM TABLE_COLUMNS "
       "WHERE TABLE_NAME = 'SALES_REFUNDS_TRAIN' AND COLUMN_NAME = 'BOOKING_DATE'"
      ).collect()  # confirm it says LONGDATE

cc.connection.cursor().execute("""
    CREATE COLUMN TABLE SALES_REFUNDS_TRAIN_FIXED AS (
        SELECT CAST("BOOKING_DATE" AS DATE) AS "BOOKING_DATE", "REFUNDS"
        FROM SALES_REFUNDS_TRAIN
    )
""")
cc.connection.cursor().execute("DROP TABLE SALES_REFUNDS_TRAIN")
cc.connection.cursor().execute(
    'RENAME TABLE SALES_REFUNDS_TRAIN_FIXED TO SALES_REFUNDS_TRAIN'
)

# Verify
print(cc.sql(
    "SELECT DATA_TYPE_NAME FROM TABLE_COLUMNS "
    "WHERE TABLE_NAME = 'SALES_REFUNDS_TRAIN' AND COLUMN_NAME = 'BOOKING_DATE'"
).collect())
# expected: DATA_TYPE_NAME == 'DATE'

## 3. Ask for an AutoML model tuned to the diagnostics

Again the user speaks plainly — \"tune to what you found\", \"faster search\". Under the hood, when the agent has a tailored search space in mind, it fetches PAL's `default`/`light` template via `get_automl_config_dict`, optionally reshapes it via `modify_automl_config_dict` (which runs `VERIFY_CONFIG=1` server-side), and hands the resolved `config_dict` straight to `automatic_timeseries_fit_and_save`.

None of that leaks into the prompt.

In [ ]:
print(chatbot.chat(
    "Use the automatic_timeseries pipeline to train a forecasting model on "
    "SALES_REFUNDS_TRAIN and save it as automl_from_ts_check. Tailor the search space "
    "to the characteristics you just discovered — do not fall back to the default "
    "template. Keep the search short (around two generations, small population) so it "
    "finishes quickly."
))

In [ ]:
print(chatbot.chat("show me the suggested AutoML config dict for this dataset and pipeline."))

In [ ]:
print(chatbot.chat("show me the suggested configdict"))

### Inspect what the agent actually did with `config_dict`

The tracer we installed at set-up time captured every call the agent made to `get_automl_config_dict`, `modify_automl_config_dict` and `automatic_timeseries_fit_and_save`. We can now see:

- which template the agent started from (`default` / `light` / `empty`),
- whether it modified the template and, if so, with what add/remove/replace/modify payload,
- whether `VERIFY_CONFIG=1` accepted the modification,
- the final `config_dict` that was passed to training.

In [ ]:
import json
from collections import Counter

print(f"Total tool calls captured: {len(tool_call_log)}")
counts = Counter(e["tool"] for e in tool_call_log)
for name, n in counts.most_common():
    print(f"  {n:>3}  {name}")

print("\n--- Full trace (order matters) ---")
for idx, entry in enumerate(tool_call_log, 1):
    print(f"\n[{idx}] {entry['tool']}")
    inputs = entry.get("inputs", {})
    # Keep the summary short but flag config_dict clearly.
    inputs_summary = {k: (v if k != "config_dict" else ("<None>" if v is None else "<config_dict provided>"))
                      for k, v in inputs.items()}
    print("    inputs :", json.dumps(inputs_summary, ensure_ascii=False, default=str))
    if "error" in entry:
        print("    ERROR  :", entry["error"])
        continue
    result = entry.get("result")
    if entry["tool"] == "get_automl_config_dict" and isinstance(result, dict):
        print("    pipeline_type:", result.get("pipeline_type"))
        ops = result.get("operators") or []
        print(f"    operators: {len(ops)} entries")
        for op in ops[:5]:
            print(f"      - {op.get('operator')}  ({op.get('type')})")
        if len(ops) > 5:
            print(f"      ... {len(ops) - 5} more")
    elif entry["tool"] == "modify_automl_config_dict" and isinstance(result, dict):
        if result.get("error"):
            print(f"    error: {result['error']}  detail: {result.get('detail')}")
        else:
            print(f"    pipeline_type: {result.get('pipeline_type')}  operators: {len(result.get('operators') or [])}")

# Zoom in on the fit call
fit_calls = [e for e in tool_call_log if e["tool"] == "automatic_timeseries_fit_and_save"]
if fit_calls:
    cfg = fit_calls[-1]["inputs"].get("config_dict")
    print("\n=== config_dict handed to automatic_timeseries_fit_and_save ===")
    if cfg is None:
        print("(no config_dict passed — agent ran the default AutoML pipeline)")
    else:
        print(json.dumps(cfg, indent=2, ensure_ascii=False, default=str))
else:
    print("\n(agent did not call automatic_timeseries_fit_and_save)")

## 4. Predict on the held-out split and plot

In [ ]:
print(chatbot.chat(
    "Use automl_from_ts_check to forecast SALES_REFUNDS_PREDICT — BOOKING_DATE is the "
    "key. Show me the first 10 predicted rows."
))

In [ ]:
print(chatbot.chat(
    "Plot the latest forecast produced by automl_from_ts_check against the actual "
    "values in SALES_REFUNDS and give me the plot file path."
))